# Baseline Isolation: Tensor Structure vs. Parameter Count

This notebook implements the controlled baseline study described in **ML_QRC-12.pdf**.

It evaluates two analytical baselines that isolate different parts of the proposed ML-QRC gain:

- **Ridge-PCA-d3**: flatten the QRC tensor, compress with PCA on the training set only, then fit Ridge.
- **Tensor-Ridge**: compress the QRC tensor directly with a Tucker/HOSVD projection on the training set only, then fit Ridge.

The purpose is to separate:

1. **flat analytical compression** without tensor structure,
2. **tensor-aware analytical compression** without gradient training,
3. the full proposed method **ML-QRC-M**, which keeps tensor structure and learns the projections by gradient descent.

This notebook uses the same cached QRC feature banks and the same train/test protocols as the main Task 1 and Task 2 notebooks.


In [ ]:
# This cell installs the small set of packages used by the notebook.

%pip install -q numpy pandas matplotlib torch scikit-learn pennylane jupyter


In [ ]:
# This cell imports the libraries and resolves the repository root.

from pathlib import Path
import os
import random

here = Path.cwd()
if (here / "scripts").exists():
    ROOT = here
elif (here.parent / "scripts").exists():
    ROOT = here.parent
else:
    ROOT = here

cache_root = ROOT / "notebooks" / "artifacts" / "cache"
cache_root.mkdir(parents=True, exist_ok=True)
(cache_root / "mplconfig").mkdir(parents=True, exist_ok=True)

os.environ["XDG_CACHE_HOME"] = str(cache_root.resolve())
os.environ["MPLCONFIGDIR"] = str((cache_root / "mplconfig").resolve())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import pennylane as qml

from sklearn.decomposition import PCA

plt.rcParams["figure.dpi"] = 1000
plt.rcParams["savefig.dpi"] = 1000
plt.rcParams["figure.figsize"] = (3, 2.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

pd.set_option("display.precision", 6)
torch.set_num_threads(1)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ARTIFACT_DIR = ROOT / "notebooks" / "artifacts" / "baseline_isolation"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print(f"Repo root: {ROOT.resolve()}")
print(f"Artifacts: {ARTIFACT_DIR.resolve()}")
print(f"PyTorch: {torch.__version__}")
print(f"Torch device: {DEVICE}")
print(f"PennyLane: {qml.__version__}")


## Important Fairness Notes

This notebook follows the draft intent, but it also corrects two mathematical edge cases:

1. **PCA to 512 dimensions is impossible when the training subset has fewer than 512 samples.**
   In that case, the notebook uses the maximum feasible PCA rank and records the effective dimension.

2. **A Tucker-(8, 8, 8) compression is impossible on the qubit mode when `n = 6`.**
   The notebook therefore caps the Tucker rank by the mode size and uses the feasible tensor ranks `(8, 6, 8)`.

These corrections make the comparison mathematically valid and fully reproducible.


In [ ]:
# This cell stores the experiment configuration and the paths to the
# cached artifacts produced by the main Task 1 and Task 2 notebooks.

CONFIG = {
    "eval_seeds": list(range(10)),
    "task1_low_data_train": 300,
    "task1_full_train": 3000,
    "task2_half_train": 1120,
    "pca_target_dim": 512,
    "tensor_target_ranks": (8, 8, 8),
    "ridge_alphas": (1e-4, 1e-3, 1e-2, 0.1, 1.0, 10.0, 100.0, 1e3, 1e4, 1e5),
}

TASK1_DIR = ROOT / "notebooks" / "artifacts" / "task1_mg_v2"
TASK2_DIR = ROOT / "notebooks" / "artifacts" / "task3_tfim"

TASK1_FEATURES = TASK1_DIR / "qrc_feats_rseed42_p0.01_q6_T50.npy"
TASK1_TABLE = TASK1_DIR / "table2_main.csv"
TASK2_FEATURES = TASK2_DIR / "qrc_feats_tfim_rseed42_p0.01_q6_T50.npy"
TASK2_TABLE = TASK2_DIR / "table_phase.csv"

required = [TASK1_FEATURES, TASK1_TABLE, TASK2_FEATURES, TASK2_TABLE]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required cached artifacts. Run the main Task 1 / Task 2 notebooks first.\n"
        + "\n".join(missing)
    )

display(pd.Series(CONFIG, name="value"))
print("Required artifacts found.")


In [ ]:
# This cell recreates the Mackey-Glass targets and splits from Task 1.

def generate_mackey_glass(num_points, beta, gamma, q, tau, step, x0):
    delay = int(round(tau / step))
    total = delay + num_points
    x = np.full(total + 1, x0, dtype=np.float64)

    def rhs(x_now, x_delay):
        return beta * x_delay / (1.0 + x_delay ** q) - gamma * x_now

    for k in range(delay, total):
        x_now = x[k]
        x_d0 = x[k - delay]
        x_dh = 0.5 * (x[k - delay] + x[k - delay + 1])
        x_d1 = x[k - delay + 1]
        k1 = rhs(x_now, x_d0)
        k2 = rhs(x_now + 0.5 * step * k1, x_dh)
        k3 = rhs(x_now + 0.5 * step * k2, x_dh)
        k4 = rhs(x_now + step * k3, x_d1)
        x[k + 1] = x_now + (step / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)

    return x[delay + 1 : delay + 1 + num_points].astype(np.float32)

mg = generate_mackey_glass(
    num_points=5000,
    beta=0.2,
    gamma=0.1,
    q=10,
    tau=17.0,
    step=0.1,
    x0=1.2,
)

T = 50
W = 1000
TR = 3000
TE = 1000

scale_min = float(mg[W:W + TR].min())
scale_range = float(mg[W:W + TR].max() - mg[W:W + TR].min())
mg_scaled = ((mg - scale_min) / scale_range).astype(np.float32)

def unscale_task1(values):
    return (np.asarray(values, dtype=np.float64) * scale_range + scale_min).astype(np.float32)

decision_times = np.arange(T, len(mg), dtype=np.int32)
task1_targets_scaled = mg_scaled[decision_times].astype(np.float32)
task1_targets_original = mg[decision_times].astype(np.float32)

task1_train_positions = np.where((decision_times >= W) & (decision_times < W + TR))[0]
task1_test_positions = np.where((decision_times >= W + TR) & (decision_times < W + TR + TE))[0]

assert len(task1_train_positions) == 3000
assert len(task1_test_positions) == 1000

task1_qrc = np.load(TASK1_FEATURES, mmap_mode="r")
print(f"Task 1 QRC tensor bank: {task1_qrc.shape}")


In [ ]:
# This cell recreates the TFIM labels and split protocol from Task 2.
# The notebook reuses the cached feature bank and reconstructs the same
# 70/30 split deterministically from the original data seed.

N_G = 16
N_SEQ = 200
N_Q = 6
T_TFIM = 50
TRAIN_FRAC = 0.70
DATA_SEED = 42

g_values = np.linspace(0.2, 1.8, N_G)
g_labels = np.array([0 if g < 1 else 1 for g in g_values], dtype=np.int64)

all_labels = np.repeat(g_labels, N_SEQ)

rng_split = np.random.default_rng(DATA_SEED + 1)
train_idx = []
test_idx = []
for gi in range(N_G):
    start = gi * N_SEQ
    idx = np.arange(start, start + N_SEQ)
    rng_split.shuffle(idx)
    n_tr = round(N_SEQ * TRAIN_FRAC)
    train_idx.extend(idx[:n_tr])
    test_idx.extend(idx[n_tr:])

train_idx = np.array(train_idx, dtype=np.int64)
test_idx = np.array(test_idx, dtype=np.int64)

y_train = all_labels[train_idx]
y_test = all_labels[test_idx]

task2_qrc_all = np.load(TASK2_FEATURES, mmap_mode="r")
task2_train_qrc = task2_qrc_all[train_idx]
task2_test_qrc = task2_qrc_all[test_idx]

N_TRAIN = len(train_idx)
N_TEST = len(test_idx)

print(f"Task 2 QRC tensor bank: {task2_qrc_all.shape}")
print(f"Task 2 train/test: {task2_train_qrc.shape} / {task2_test_qrc.shape}")
print(f"Task 2 FM/PM in train: {(y_train==0).sum()} / {(y_train==1).sum()}")


In [ ]:
# This cell defines the shared utilities for standardization,
# regression/classification Ridge, PCA compression, and Tucker/HOSVD compression.

def nmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2))

def accuracy(y_true, y_pred):
    return 100.0 * float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))

class Standardizer:
    def fit(self, X):
        flat = X.reshape(X.shape[0], -1).astype(np.float64)
        self.mu = flat.mean(axis=0)
        self.sig = flat.std(axis=0)
        self.sig[self.sig < 1e-8] = 1.0
        return self

    def transform(self, X):
        shape = X.shape
        flat = X.reshape(shape[0], -1).astype(np.float64)
        flat = (flat - self.mu) / self.sig
        return flat.reshape(shape).astype(np.float32)

def fit_ols(Xtr, ytr):
    design = np.hstack([Xtr.astype(np.float64), np.ones((len(Xtr), 1), dtype=np.float64)])
    weights, _, _, _ = np.linalg.lstsq(design, ytr.astype(np.float64), rcond=None)
    return weights.astype(np.float32)

def predict_ols(weights, X):
    design = np.hstack([X.astype(np.float64), np.ones((len(X), 1), dtype=np.float64)])
    return (design @ weights.astype(np.float64)).astype(np.float32)

def fit_ridge_regression_cv(Xtr, ytr, alphas, val_frac=0.2):
    split = max(1, int(len(Xtr) * (1.0 - val_frac)))
    Xt = Xtr[:split].astype(np.float64)
    yt = ytr[:split].astype(np.float64)
    Xv = Xtr[split:].astype(np.float64)
    yv = ytr[split:].astype(np.float64)

    mean_x = Xt.mean(axis=0)
    mean_y = yt.mean()
    centered_x = Xt - mean_x
    centered_y = yt - mean_y

    best_alpha = None
    best_score = np.inf

    for alpha in alphas:
        dual = np.linalg.solve(
            centered_x @ centered_x.T + alpha * np.eye(len(Xt)),
            centered_y,
        )
        weights = centered_x.T @ dual
        bias = mean_y - mean_x @ weights
        score = nmse(yv, Xv @ weights + bias)
        if score < best_score:
            best_score = score
            best_alpha = alpha

    full_mean_x = Xtr.astype(np.float64).mean(axis=0)
    full_mean_y = float(ytr.mean())
    full_centered_x = Xtr.astype(np.float64) - full_mean_x
    full_centered_y = ytr.astype(np.float64) - full_mean_y
    dual = np.linalg.solve(
        full_centered_x @ full_centered_x.T + best_alpha * np.eye(len(Xtr)),
        full_centered_y,
    )
    weights = full_centered_x.T @ dual
    bias = full_mean_y - full_mean_x @ weights
    return np.append(weights, bias).astype(np.float32), best_alpha

def fit_ridge_binary_cv(Xtr, ytr, alphas, val_frac=0.2):
    split = max(1, int(len(Xtr) * (1.0 - val_frac)))
    Xt = Xtr[:split].astype(np.float64)
    yt = ytr[:split].astype(np.float64)
    Xv = Xtr[split:].astype(np.float64)
    yv = ytr[split:]

    mean_x = Xt.mean(axis=0)
    centered_x = Xt - mean_x

    best_alpha = None
    best_acc = -1.0

    for alpha in alphas:
        dual = np.linalg.solve(
            centered_x @ centered_x.T + alpha * np.eye(len(Xt)),
            yt,
        )
        weights = centered_x.T @ dual
        predictions = ((Xv - mean_x) @ weights >= 0.5).astype(int)
        acc = float(np.mean(predictions == yv))
        if acc > best_acc:
            best_acc = acc
            best_alpha = alpha

    full_mean_x = Xtr.astype(np.float64).mean(axis=0)
    full_centered_x = Xtr.astype(np.float64) - full_mean_x
    dual = np.linalg.solve(
        full_centered_x @ full_centered_x.T + best_alpha * np.eye(len(Xtr)),
        ytr.astype(np.float64),
    )
    weights = full_centered_x.T @ dual
    bias = float(ytr.mean() - full_mean_x @ weights)
    return weights.astype(np.float32), bias, best_alpha

def predict_binary_linear(w, b, X):
    return (X.astype(np.float64) @ w.astype(np.float64) + b >= 0.5).astype(int)

def fit_pca_projection(Xtr_flat, target_dim, seed=0):
    effective_dim = min(target_dim, Xtr_flat.shape[0], Xtr_flat.shape[1])
    pca = PCA(
        n_components=effective_dim,
        svd_solver="randomized",
        random_state=seed,
    )
    pca.fit(Xtr_flat.astype(np.float64))
    return pca, effective_dim

def unfold_mode(X, mode):
    if mode == 0:
        return np.transpose(X, (1, 0, 2, 3)).reshape(X.shape[1], -1)
    if mode == 1:
        return np.transpose(X, (2, 0, 1, 3)).reshape(X.shape[2], -1)
    if mode == 2:
        return np.transpose(X, (3, 0, 1, 2)).reshape(X.shape[3], -1)
    raise ValueError("mode must be 0, 1, or 2")

def fit_hosvd_factors(Xtr_tensor, target_ranks):
    shape = Xtr_tensor.shape[1:]
    effective_ranks = tuple(min(target_ranks[i], shape[i]) for i in range(3))
    factors = []
    for mode, rank in enumerate(effective_ranks):
        unfolded = unfold_mode(Xtr_tensor, mode)
        U, _, _ = np.linalg.svd(unfolded, full_matrices=False)
        factors.append(U[:, :rank].astype(np.float32))
    return factors, effective_ranks

def apply_tucker_projection(X, factors):
    time_basis, qubit_basis, obs_basis = factors
    Y = np.einsum("btnf,ta->banf", X, time_basis)
    Y = np.einsum("banf,nc->bacf", Y, qubit_basis)
    Y = np.einsum("bacf,fd->bacd", Y, obs_basis)
    return Y.astype(np.float32)

print("Helpers ready.")


In [ ]:
# This cell runs the Task 1 baseline-isolation experiment on both
# the full-data and 10% regimes.

task1_reference = pd.read_csv(TASK1_TABLE)
task1_ref_map = {}
for _, row in task1_reference.iterrows():
    key = row["raw_key"]
    task1_ref_map[key] = {
        "full_mean": float(row["full_mean"]),
        "full_std": float(row["full_std"]),
        "low_data_mean": float(row["low_data_mean"]),
        "low_data_std": float(row["low_data_std"]),
    }

task1_rows = []
task1_detail_rows = []

task1_regimes = {
    "full": CONFIG["task1_full_train"],
    "low_data": CONFIG["task1_low_data_train"],
}

for regime, n_tr in task1_regimes.items():
    train_pos = task1_train_positions[:n_tr]
    scaler = Standardizer().fit(task1_qrc[train_pos])
    Xtr_t = scaler.transform(task1_qrc[train_pos])
    Xte_t = scaler.transform(task1_qrc[task1_test_positions])
    Xtr_f = Xtr_t.reshape(len(train_pos), -1)
    Xte_f = Xte_t.reshape(len(task1_test_positions), -1)
    ytr = task1_targets_scaled[train_pos]
    yte = task1_targets_original[task1_test_positions]

    ridge_weights, ridge_alpha = fit_ridge_regression_cv(Xtr_f, ytr, CONFIG["ridge_alphas"])
    ridge_score = nmse(yte, unscale_task1(predict_ols(ridge_weights, Xte_f)))

    pca, pca_dim = fit_pca_projection(Xtr_f, CONFIG["pca_target_dim"], seed=0)
    Xtr_pca = pca.transform(Xtr_f.astype(np.float64)).astype(np.float32)
    Xte_pca = pca.transform(Xte_f.astype(np.float64)).astype(np.float32)
    pca_ridge_weights, pca_ridge_alpha = fit_ridge_regression_cv(Xtr_pca, ytr, CONFIG["ridge_alphas"])
    ridge_pca_score = nmse(yte, unscale_task1(predict_ols(pca_ridge_weights, Xte_pca)))

    factors, effective_ranks = fit_hosvd_factors(Xtr_t, CONFIG["tensor_target_ranks"])
    Xtr_tucker = apply_tucker_projection(Xtr_t, factors).reshape(len(train_pos), -1)
    Xte_tucker = apply_tucker_projection(Xte_t, factors).reshape(len(task1_test_positions), -1)
    tensor_ridge_weights, tensor_ridge_alpha = fit_ridge_regression_cv(Xtr_tucker, ytr, CONFIG["ridge_alphas"])
    tensor_ridge_score = nmse(yte, unscale_task1(predict_ols(tensor_ridge_weights, Xte_tucker)))

    task1_detail_rows.extend(
        [
            {
                "regime": regime,
                "method": "Ridge",
                "score": ridge_score,
                "effective_dim": Xtr_f.shape[1],
                "extra": f"alpha={ridge_alpha}",
            },
            {
                "regime": regime,
                "method": "Ridge-PCA-d3",
                "score": ridge_pca_score,
                "effective_dim": int(pca_dim),
                "extra": f"alpha={pca_ridge_alpha}",
            },
            {
                "regime": regime,
                "method": "Tensor-Ridge",
                "score": tensor_ridge_score,
                "effective_dim": int(np.prod(effective_ranks)),
                "extra": f"ranks={effective_ranks}, alpha={tensor_ridge_alpha}",
            },
        ]
    )

task1_task_rows = [
    {
        "Method": "Ridge",
        "full_mean": task1_detail_rows[0]["score"],
        "full_std": 0.0,
        "low_data_mean": task1_detail_rows[3]["score"],
        "low_data_std": 0.0,
    },
    {
        "Method": "Ridge-PCA-d3",
        "full_mean": task1_detail_rows[1]["score"],
        "full_std": 0.0,
        "low_data_mean": task1_detail_rows[4]["score"],
        "low_data_std": 0.0,
    },
    {
        "Method": "Tensor-Ridge",
        "full_mean": task1_detail_rows[2]["score"],
        "full_std": 0.0,
        "low_data_mean": task1_detail_rows[5]["score"],
        "low_data_std": 0.0,
    },
    {
        "Method": "ML-QRC (ours)",
        "full_mean": task1_ref_map["ML-QRC-M"]["full_mean"],
        "full_std": task1_ref_map["ML-QRC-M"]["full_std"],
        "low_data_mean": task1_ref_map["ML-QRC-M"]["low_data_mean"],
        "low_data_std": task1_ref_map["ML-QRC-M"]["low_data_std"],
    },
]
task1_baseline_table = pd.DataFrame(task1_task_rows)
task1_detail_table = pd.DataFrame(task1_detail_rows)

display(task1_baseline_table)
display(task1_detail_table)


In [ ]:
# This cell runs the Task 2 baseline-isolation experiment on the
# full-data and 50% regimes.

task2_reference = pd.read_csv(TASK2_TABLE)
task2_ref_map = {}
for _, row in task2_reference.iterrows():
    task2_ref_map[row["Method"]] = {
        "full_mean": float(row["full_mean"]),
        "full_std": float(row["full_std"]),
        "half_data_mean": float(row["half_data_mean"]),
        "half_data_std": float(row["half_data_std"]),
    }

task2_full_rows = []
task2_half_rows = []

task2_regimes = {
    "full": N_TRAIN,
    "half_data": CONFIG["task2_half_train"],
}

for regime, n_tr in task2_regimes.items():
    regime_scores = {"Ridge": [], "Ridge-PCA-d3": [], "Tensor-Ridge": []}
    regime_meta = []

    if regime == "full":
        seed_list = [0]
    else:
        seed_list = CONFIG["eval_seeds"]

    for seed in seed_list:
        if regime == "full":
            train_subset = np.arange(len(y_train), dtype=np.int64)
        else:
            rng_sub = np.random.default_rng(seed)
            fm_idx = np.where(y_train == 0)[0]
            pm_idx = np.where(y_train == 1)[0]
            half = n_tr // 2
            sub_fm = rng_sub.choice(fm_idx, size=half, replace=False)
            sub_pm = rng_sub.choice(pm_idx, size=half, replace=False)
            train_subset = np.concatenate([sub_fm, sub_pm])
            rng_sub.shuffle(train_subset)

        scaler = Standardizer().fit(task2_train_qrc[train_subset])
        Xtr_t = scaler.transform(task2_train_qrc[train_subset])
        Xte_t = scaler.transform(task2_test_qrc)
        Xtr_f = Xtr_t.reshape(len(train_subset), -1)
        Xte_f = Xte_t.reshape(len(y_test), -1)
        ytr = y_train[train_subset]

        ridge_w, ridge_b, ridge_alpha = fit_ridge_binary_cv(Xtr_f, ytr, CONFIG["ridge_alphas"])
        ridge_score = accuracy(y_test, predict_binary_linear(ridge_w, ridge_b, Xte_f))

        pca, pca_dim = fit_pca_projection(Xtr_f, CONFIG["pca_target_dim"], seed=seed)
        Xtr_pca = pca.transform(Xtr_f.astype(np.float64)).astype(np.float32)
        Xte_pca = pca.transform(Xte_f.astype(np.float64)).astype(np.float32)
        pca_w, pca_b, pca_alpha = fit_ridge_binary_cv(Xtr_pca, ytr, CONFIG["ridge_alphas"])
        ridge_pca_score = accuracy(y_test, predict_binary_linear(pca_w, pca_b, Xte_pca))

        factors, effective_ranks = fit_hosvd_factors(Xtr_t, CONFIG["tensor_target_ranks"])
        Xtr_tucker = apply_tucker_projection(Xtr_t, factors).reshape(len(train_subset), -1)
        Xte_tucker = apply_tucker_projection(Xte_t, factors).reshape(len(y_test), -1)
        tensor_w, tensor_b, tensor_alpha = fit_ridge_binary_cv(Xtr_tucker, ytr, CONFIG["ridge_alphas"])
        tensor_score = accuracy(y_test, predict_binary_linear(tensor_w, tensor_b, Xte_tucker))

        regime_scores["Ridge"].append(ridge_score)
        regime_scores["Ridge-PCA-d3"].append(ridge_pca_score)
        regime_scores["Tensor-Ridge"].append(tensor_score)
        regime_meta.append(
            {
                "regime": regime,
                "seed": seed,
                "ridge_alpha": ridge_alpha,
                "pca_dim": int(pca_dim),
                "pca_alpha": pca_alpha,
                "tensor_ranks": str(effective_ranks),
                "tensor_dim": int(np.prod(effective_ranks)),
                "tensor_alpha": tensor_alpha,
            }
        )

    if regime == "full":
        task2_full_rows = regime_scores
        task2_full_meta = pd.DataFrame(regime_meta)
    else:
        task2_half_rows = regime_scores
        task2_half_meta = pd.DataFrame(regime_meta)

task2_task_rows = [
    {
        "Method": "Ridge",
        "full_mean": float(np.mean(task2_full_rows["Ridge"])),
        "full_std": float(np.std(task2_full_rows["Ridge"])),
        "half_data_mean": float(np.mean(task2_half_rows["Ridge"])),
        "half_data_std": float(np.std(task2_half_rows["Ridge"])),
    },
    {
        "Method": "Ridge-PCA-d3",
        "full_mean": float(np.mean(task2_full_rows["Ridge-PCA-d3"])),
        "full_std": float(np.std(task2_full_rows["Ridge-PCA-d3"])),
        "half_data_mean": float(np.mean(task2_half_rows["Ridge-PCA-d3"])),
        "half_data_std": float(np.std(task2_half_rows["Ridge-PCA-d3"])),
    },
    {
        "Method": "Tensor-Ridge",
        "full_mean": float(np.mean(task2_full_rows["Tensor-Ridge"])),
        "full_std": float(np.std(task2_full_rows["Tensor-Ridge"])),
        "half_data_mean": float(np.mean(task2_half_rows["Tensor-Ridge"])),
        "half_data_std": float(np.std(task2_half_rows["Tensor-Ridge"])),
    },
    {
        "Method": "ML-QRC (ours)",
        "full_mean": task2_ref_map["ML-QRC (ours)"]["full_mean"],
        "full_std": task2_ref_map["ML-QRC (ours)"]["full_std"],
        "half_data_mean": task2_ref_map["ML-QRC (ours)"]["half_data_mean"],
        "half_data_std": task2_ref_map["ML-QRC (ours)"]["half_data_std"],
    },
]

task2_baseline_table = pd.DataFrame(task2_task_rows)
task2_meta = pd.concat([task2_full_meta, task2_half_meta], ignore_index=True)

display(task2_baseline_table)
display(task2_meta.head(12))


In [ ]:
# This cell saves the paper-ready CSV outputs.

task1_csv = ARTIFACT_DIR / "task1_baseline_isolation.csv"
task2_csv = ARTIFACT_DIR / "task2_baseline_isolation.csv"
task1_detail_csv = ARTIFACT_DIR / "task1_baseline_isolation_detail.csv"
task2_meta_csv = ARTIFACT_DIR / "task2_baseline_isolation_detail.csv"

task1_baseline_table.to_csv(task1_csv, index=False, float_format="%.6f")
task2_baseline_table.to_csv(task2_csv, index=False, float_format="%.6f")
task1_detail_table.to_csv(task1_detail_csv, index=False)
task2_meta.to_csv(task2_meta_csv, index=False)

print(f"Saved: {task1_csv.resolve()}")
print(f"Saved: {task2_csv.resolve()}")
print(f"Saved: {task1_detail_csv.resolve()}")
print(f"Saved: {task2_meta_csv.resolve()}")


In [ ]:
# This cell prints compact paper-friendly summaries for both tasks.

print("=" * 88)
print(" Task 1  Baseline isolation on Mackey-Glass  (NMSE ↓)")
print("=" * 88)
print(f"  {'Method':<18} {'Full (3000)':>20}   {'10% (300)':>20}")
print("-" * 88)
for _, row in task1_baseline_table.iterrows():
    print(
        f"  {row['Method']:<18}  "
        f"{row['full_mean']:.4f} ± {row['full_std']:.4f}   "
        f"{row['low_data_mean']:.4f} ± {row['low_data_std']:.4f}"
    )
print("=" * 88)

print("\n" + "=" * 88)
print(" Task 2  Baseline isolation on TFIM  (Accuracy % ↑)")
print("=" * 88)
print(f"  {'Method':<18} {'Full (2240)':>20}   {'50% (1120)':>20}")
print("-" * 88)
for _, row in task2_baseline_table.iterrows():
    print(
        f"  {row['Method']:<18}  "
        f"{row['full_mean']:.1f} ± {row['full_std']:.1f}   "
        f"{row['half_data_mean']:.1f} ± {row['half_data_std']:.1f}"
    )
print("=" * 88)


In [ ]:
# This cell makes a compact Task 1 comparison figure.

fig, axes = plt.subplots(1, 2, figsize=(6, 2.5), dpi=1000)
colors = ["#74c476", "#9ecae1", "#3182bd", "#08519c"]

for ax, reg, title in zip(
    axes,
    ["full", "low_data"],
    ["Task 1 full", "Task 1 low-data"],
):
    if reg == "full":
        means = task1_baseline_table["full_mean"].to_numpy()
        stds = task1_baseline_table["full_std"].to_numpy()
    else:
        means = task1_baseline_table["low_data_mean"].to_numpy()
        stds = task1_baseline_table["low_data_std"].to_numpy()

    x = np.arange(len(task1_baseline_table))
    ax.bar(x, means, yerr=stds, width=0.58, color=colors, capsize=3)
    ax.set_xticks(x)
    ax.set_xticklabels(task1_baseline_table["Method"], rotation=40, ha="right", fontsize=6)
    ax.set_ylabel("NMSE")
    ax.set_title(title, fontsize=8)

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "fig_baseline_isolation_task1.pdf", bbox_inches="tight")
plt.savefig(ARTIFACT_DIR / "fig_baseline_isolation_task1.png", bbox_inches="tight")
plt.show()
print("Saved: fig_baseline_isolation_task1.pdf")


In [ ]:
# This cell makes a compact Task 2 comparison figure.

fig, axes = plt.subplots(1, 2, figsize=(6, 2.5), dpi=1000)
colors = ["#74c476", "#9ecae1", "#3182bd", "#08519c"]

for ax, reg, title in zip(
    axes,
    ["full", "half_data"],
    ["Task 2 full", "Task 2 half-data"],
):
    if reg == "full":
        means = task2_baseline_table["full_mean"].to_numpy()
        stds = task2_baseline_table["full_std"].to_numpy()
    else:
        means = task2_baseline_table["half_data_mean"].to_numpy()
        stds = task2_baseline_table["half_data_std"].to_numpy()

    x = np.arange(len(task2_baseline_table))
    ax.bar(x, means, yerr=stds, width=0.58, color=colors, capsize=3)
    ax.set_xticks(x)
    ax.set_xticklabels(task2_baseline_table["Method"], rotation=40, ha="right", fontsize=6)
    ax.set_ylabel("Accuracy (%)")
    ax.set_title(title, fontsize=8)

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "fig_baseline_isolation_task2.pdf", bbox_inches="tight")
plt.savefig(ARTIFACT_DIR / "fig_baseline_isolation_task2.png", bbox_inches="tight")
plt.show()
print("Saved: fig_baseline_isolation_task2.pdf")


## Summary

This notebook implements the **Baseline Isolation: Tensor Structure vs. Parameter Count** study requested by the draft.

It reuses the cached QRC feature banks from the main notebooks, computes:

- `Ridge`,
- `Ridge-PCA-d3`,
- `Tensor-Ridge`,
- and compares them to the existing `ML-QRC (ours)` reference rows.

The saved outputs are:

- `task1_baseline_isolation.csv`
- `task2_baseline_isolation.csv`
- `fig_baseline_isolation_task1.pdf`
- `fig_baseline_isolation_task2.pdf`
